In [1]:
import os

import numpy as np
import pandas as pd

from collections import Counter

import seaborn as sns
import matplotlib.pyplot as plt

from rdkit import Chem

from routehunter_build.paper import get_work_by_doi

In [2]:
sheet_id_oprd = "18aP203JdmhgdD67P-vTjparpQqh90lN80RjOaIGSyKs"
sheet_id_other = "1p-n7Y03KD8fkRSwpkFf5hNBKzr4Kj3cMyh6YpLulnXE"
#
url_oprd = f"https://docs.google.com/spreadsheets/d/{sheet_id_oprd}/export?format=csv"
url_other = f"https://docs.google.com/spreadsheets/d/{sheet_id_other}/export?format=csv"

### 1. Collection progress

In [ ]:
tmp = pd.concat([pd.read_csv(url_oprd), pd.read_csv(url_other)])
tot = len(tmp)
print(f"Total papers:     {tot}")
tmp = tmp.dropna()
proc = len(tmp)
print(f"Processed papers: {proc} ({100*proc/tot:.1f}%)")
tmp = tmp[tmp["target"] != "SKIP"]
print(f"Collected targets:   {len(tmp)}")

### 2. Check smiles

In [ ]:
target_df = pd.concat([pd.read_csv(url_oprd), pd.read_csv(url_other)]).dropna()
target_df = target_df[target_df["target"] != "SKIP"]

# smiles check
for row in target_df.iterrows():
    mol = Chem.MolFromSmiles(row[1]["target"])
    if not mol:
        raise("Invalid SMILES")
    inchikey = Chem.MolToInchiKey(mol)

### 3. Data subsets preparation

In [ ]:
api_key = "Q2bbP41q4woiJUZBbtVSpU"

In [ ]:
target_df = pd.concat([
    pd.read_csv(url_oprd).dropna(subset="target"), 
    pd.read_csv(url_other).dropna(subset="target")]
)
target_df.head()

In [ ]:
res = []
for n, (idx, paper) in enumerate(target_df.iterrows()):

    print(f"{n + 1} / {len(target_df)}", end="\r")

    # 1. Get meta data
    meta = get_work_by_doi(paper["doi"], api_key=api_key)

    # 2. Create res
    data_point = {
         
        "journal": meta["venue"],
        "title": meta["title"], 
        "abstract": meta["abstract"], 
        "year": meta["publication_year"],
        "doi": paper["doi"],
        "target": paper["target"],
        "contributor": "Dmitry Zankov",
    }

    # 3. Get route
    if paper["target"] == "SKIP":
        data_point["has_route"] = 0
    else:
        data_point["has_route"] = 1

    # 4. Append data point
    res.append(data_point)
#
df_main = pd.DataFrame(res)

In [ ]:
# 1. Row dataset
df_main.to_csv("rh-data/core/routehunter_main.csv", index=False)

# 2. Seed dataset
df_seed = df_main.copy()[df_main["has_route"] == 1]
df_seed = df_seed.drop("has_route", axis=1)
df_seed.to_csv("rh-data/core/routehunter_seed.csv", index=False)

# 3. Route prob dataset
df_route = df_main.copy().drop_duplicates(subset="doi")
df_route = df_route[["title", "abstract", "doi", "has_route"]]
df_route = df_route.dropna(subset="title")
df_route.to_csv("rh-data/training/abstract_data.csv", index=False)

### 4. CASP data amd model

In [ ]:
import pandas as pd
from rdkit import Chem

from sklearn.model_selection import train_test_split
from routehunter_build.casp import load_casp_data, train_casp_model, save_casp_model
from sklearn.metrics import balanced_accuracy_score

from routehunter_build.merge import merge_tool_tables

#### 4.1 Train solvability model

In [ ]:
az_data = pd.read_json("casp/aizynth_oprd.json", orient="table")[["target", "is_solved"]]
sp_data = pd.read_csv("casp/synplan_oprd/tree_search_stats.csv")[["target_smiles", "solved"]]
#
az_data.columns = ["smiles", "is_solved"]
sp_data.columns = ["smiles", "is_solved"]
#
az_data.to_csv("rh-data/training/az_data.csv", index=False)
sp_data.to_csv("rh-data/training/sp_data.csv", index=False)

##### AiZynthFinder model

In [ ]:
smiles, y = load_casp_data("rh-data/training/az_data.csv")
model = train_casp_model(smiles, y, random_state=42)
save_casp_model(model, output_path="rh-ata/model/az_model.pickle")

##### SynPlanner model

In [ ]:
smiles, y = load_casp_data("rh-data/training/sp_data.csv")
model = train_casp_model(smiles, y, random_state=42)
save_casp_model(model, output_path="rh-data/model/sp_model.pickle")

#### 4.2 Create solvability table

In [ ]:
df_merged = merge_tool_tables(
    {
     "AiZynthFinder":"rh-data/training/az_data.csv", 
     "SynPlanner": "rh-data/training/sp_data.csv"
    }
)
df_merged.to_csv("rhdata/core/routehunter_casp.csv", index=False)

### 5. Paper route probability

In [3]:
import pandas as pd
from collections import Counter

from routehunter_build.models import load_abstract_data, train_abstract_model, combine_text, find_threshold_for_precision

In [10]:
training_data_path = "rh-data/training/abstract_data.csv"
candidate_papers_path = "extraction_data/openalex_chem_metadata_1.csv"
monitor_high_path = "rh-data/monitor/paper_route_prob_high.csv"
monitor_medium_path = "rh-data/monitor/paper_route_prob_medium.csv"

TARGET_PRECISION_HIGH = 0.9
TARGET_PRECISION_MEDIUM = 0.8

In [7]:
text, y = load_abstract_data(training_data_path)
print(f"Loaded {len(text)} labeled papers ({sum(y)} with a route, {len(y) - sum(y)} without)")

model, meta = train_abstract_model(text, y, random_state=42)
print(f"Model metrics (5x5-CV): precision={meta['precision']:.3f}  recall={meta['recall']:.3f}  f1={meta['f1']:.3f}")

Loaded 2533 labeled papers (1287 with a route, 1246 without)
Model metrics (5x5-CV): precision=0.821  recall=0.856  f1=0.838


In [8]:
high = find_threshold_for_precision(meta["y_val"], meta["y_prob"], TARGET_PRECISION_HIGH)
medium = find_threshold_for_precision(meta["y_val"], meta["y_prob"], TARGET_PRECISION_MEDIUM)

print(f"High threshold:   {high.threshold:.3f}  precision={high.precision:.3f}  recall={high.recall:.3f}")
print(f"Medium threshold: {medium.threshold:.3f}  precision={medium.precision:.3f}  recall={medium.recall:.3f}")

threshold_high = high.threshold
threshold_medium = medium.threshold

High threshold:   0.678  precision=0.900  recall=0.637
Medium threshold: 0.466  precision=0.800  recall=0.886


In [11]:
candidates_df = pd.read_csv(candidate_papers_path)
print(f"Loaded {len(candidates_df)} candidate papers")

candidate_text = [combine_text(t, a) for t, a in zip(candidates_df["title"], candidates_df["abstract"])]
candidates_df["route_prob"] = model.predict_proba(candidate_text)[:, 1]

columns = ["journal", "title", "abstract", "doi", "publication_date", "route_prob"]
candidates_df = candidates_df[columns]
print(f"Scored {len(candidates_df)} candidate papers")

Loaded 654806 candidate papers
Scored 654806 candidate papers


In [20]:
high_df = candidates_df[candidates_df["route_prob"] >= threshold_high].sort_values("route_prob", ascending=False)
medium_df = candidates_df[candidates_df["route_prob"] >= threshold_medium].sort_values("route_prob", ascending=False)

n_below = len(candidates_df) - len(medium_df)
print(f"High   monitor (route_prob >= {threshold_high:.3f}): {len(high_df)} papers")
print(f"Medium monitor (route_prob >= {threshold_medium:.3f}): {len(medium_df)} papers")

High   monitor (route_prob >= 0.678): 22637 papers
Medium monitor (route_prob >= 0.466): 132856 papers


In [25]:
Counter(high_df["journal"])

Counter({'Tetrahedron': 7826,
         'Organic Letters': 3247,
         'Journal of Organic Chemistry': 3179,
         'Synthesis': 1846,
         'Organic Process Research & Development': 1532,
         'Synlett': 1473,
         'Journal of the American Chemical Society': 1210,
         'Angewandte Chemie International Edition': 1169,
         'European Journal of Organic Chemistry': 986,
         'Chemical Science': 107,
         'Green Chemistry': 47,
         'Reaction Chemistry & Engineering': 15})

In [21]:
high_df.to_csv(monitor_high_path, index=False)
medium_df.to_csv(monitor_medium_path, index=False)
print(f"Saved {monitor_high_path}")
print(f"Saved {monitor_medium_path}")

Saved rh-data/monitor/paper_route_prob_high.csv
Saved rh-data/monitor/paper_route_prob_medium.csv
